# Trabalho Prático de Mineração de Dados — Fase 2
## Utilização de LLMs e Análise Comparativa com Diretrizes de IA Responsável

**Disciplina:** Mineração de Dados  
**Grupo:** Grupo 13 - Arthur Faria 2024013915, Felipe Pereira 2024080051, Giordano[sobrenome] [matricula], João Vaz[matricula]

**Dataset:**  Online Retail II UCI (https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data)  

**Tarefa de mineração:** Padrões frequentes  
**Data da entrega:** 03/05/2026

---

## Objetivo deste notebook

Este notebook documenta a Fase 2 do Trabalho Prático. O objetivo é usar uma ou mais LLMs como apoio à construção da solução de mineração de dados e comparar duas trilhas de interação:

- **Trilha A — baseline:** interação com a LLM sem explicitar diretrizes de IA responsável;
- **Trilha B — guiada:** interação com a LLM explicitando as diretrizes de IA responsável escolhidas pelo grupo.

O foco não é apenas mostrar prompts e respostas. O grupo deve analisar criticamente se a explicitação das diretrizes levou a diferenças concretas na solução proposta, por exemplo em:

- escolha de algoritmos;
- tratamento dos dados;
- seleção de atributos;
- definição de parâmetros;
- métricas de avaliação;
- interpretabilidade;
- privacidade;
- análise de viés;
- custo computacional;
- qualidade da documentação.

> **Importante:** este notebook é um modelo de estrutura. Ele contém exemplos, placeholders e código genérico. O grupo deve substituir os campos indicados por informações reais do seu dataset, das interações com LLMs e dos experimentos efetivamente realizados.


# Como usar este notebook

Este notebook foi organizado seguindo a estrutura esperada para a Fase 2:

1. **Business Understanding**
2. **Data Understanding & Data Preparation**
3. **Modeling**
4. **Evaluation**
5. **Checklist final**

Ao longo do notebook, os trechos marcados entre colchetes, como `[INSERIR LINK DA CONVERSA]`, devem ser preenchidos pelo grupo.

## Diferença entre exemplo e entrega real

- Células de **exemplo** mostram como organizar a entrega.
- Células com **placeholders** indicam pontos que devem ser preenchidos.
- Células de **código** são genéricas e podem precisar de adaptação ao dataset.
- Nenhum resultado experimental deve ser inventado.
- Toda conclusão empírica deve estar apoiada em uma tabela, gráfico ou execução de código.


# 0. Controle da entrega e rastreabilidade

Esta seção registra informações mínimas para tornar o trabalho rastreável e auditável.

| Item | Valor |
|---|---|
| Nome do dataset | Online Retail II UCI |
| Link público do dataset | https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data |
| Número de linhas | 1,067,371 |
| Número de colunas | 8 |
| Tarefa de mineração | Padrões frequentes |
| LLM usada na Trilha A | [INSERIR MODELO] |
| LLM usada na Trilha B | [INSERIR MODELO] |
| Link da conversa — Trilha A | [INSERIR LINK DA CONVERSA] |
| Link da conversa — Trilha B | [INSERIR LINK DA CONVERSA] |

## Diretrizes de IA responsável escolhidas

| Diretriz | Justificativa da escolha | Como será explicitada na Trilha B |
|---|---|---|
| **Eficiência e escalabilidade** | O dataset é extenso (mais de 1 milhão de linhas considerando os dois anos). Algoritmos de mineração de regras de associação (como o Apriori) possuem alta complexidade, otimizar o tempo de processamento é um desafio crítico. | Exigiremos no prompt que a LLM proponha algoritmos computacionalmente mais eficientes (ex: FP-Growth em vez do Apriori padrão) e sugira técnicas de pré-processamento para economizar memória e otimizar o tempo de execução. |
| **Usabilidade e acessibilidade** | Para agregar valor ao negócio, os resultados precisam ser facilmente compreendidos por perfis não técnicos (como times de vendas e marketing), facilitando a tomada de decisão. | Solicitaremos no prompt que a LLM estruture o notebook de forma didática, proponha visualizações gráficas intuitivas para as regras  e traduza as métricas estatísticas em *insights* práticos de negócios. |

> **Propósito pedagógico:** esta seção evita que a análise fique solta ou apenas narrativa. O avaliador precisa conseguir identificar o que foi perguntado à LLM, quando, em qual modelo e com qual objetivo.


In [1]:
# Configurações iniciais do notebook
# Esta célula centraliza imports e parâmetros gerais.
# O grupo deve adaptar os caminhos, nomes de colunas e tipo de tarefa.

import os
import time
import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Caminho para o dataset.
# Exemplos:
# DATA_PATH = "dados/dataset.csv"
# DATA_PATH = "/content/dataset.csv"
DATA_PATH = "online_retail_II.csv"

# Tipo de tarefa do TP.
# Opções sugeridas:
# - "padroes_frequentes"
# - "agrupamento"
# - "classificacao"
TASK_TYPE = "padroes_frequentes"

# Coluna-alvo, se aplicável.
# Para padrões frequentes ou agrupamento, normalmente pode ser None.
TARGET_COLUMN = None  # Exemplo: "classe"

# Coluna ou conjunto de colunas usadas para gerar transações, se o TP for de padrões frequentes.
TRANSACTION_COLUMN = "Invoice"

ITEM_COLUMN = "Description"

print("Configuração carregada.")
print(f"Tarefa definida: {TASK_TYPE}")


Configuração carregada.
Tarefa definida: padroes_frequentes


# 1. Business Understanding

Nesta seção, o grupo deve explicar o problema, o contexto dos dados, a relevância da tarefa e a hipótese experimental sobre o uso das diretrizes de IA responsável.

A seção deve responder:

1. Qual problema será investigado?
2. Por que o dataset é relevante?
3. Qual tarefa de mineração de dados será executada?
4. Qual é o valor esperado da análise?
5. Quais diretrizes de IA responsável foram escolhidas?
6. Por que essas diretrizes são pertinentes ao problema?
7. O que se espera que mude quando essas diretrizes são explicitadas à LLM?


## 1.1 Descrição do problema de negócio

- **Contexto:** Analisamos os dados de uma loja de varejo online sediada no Reino Unido, cujos registros cobrem o período entre 2009 e 2011. A empresa vende principalmente presentes exclusivos para todas as ocasiões e possui uma base considerável de clientes atacadistas.

- **Problema:** A empresa possui um grande volume de dados transacionais, mas precisa extrair valor estratégico dessas informações. O desafio é identificar padrões de compras conjuntas para entender o comportamento do consumidor ("quais itens costumam ser comprados juntos no mesmo carrinho?").

- **Possíveis interessados:** Equipes de Marketing, Vendas e Gestão de Estoque/Logística da empresa.

- **Decisões apoiadas pela análise:** A descoberta dessas regras permitirá a criação de sistemas de recomendação precisos ("Clientes que compraram isso também compraram..."), formulação de campanhas de marketing casado, otimização da disposição de produtos no site e melhor planejamento de estoque de itens altamente correlacionados.

- **Limitações iniciais conhecidas:** O dataset possui um volume massivo de dados (mais de 1 milhão de linhas), o que exige alto poder computacional para algoritmos de mineração. Além disso, há ruídos nos dados, como notas fiscais de cancelamento (iniciadas com 'C') e valores nulos ou anômalos (como quantidades negativas) que exigirão tratamento.



## 1.2 Objetivo do dataset, origem e características gerais

### Objetivo do dataset

O dataset tem o propósito de registrar o histórico transacional da loja virtual, capturando cada item adicionado a uma nota fiscal (carrinho de compras). Ele é amplamente utilizado para testar modelos de análise de clientes, segmentação e sistemas de recomendação.

### Origem dos dados

Os dados foram originalmente doados para o UCI Machine Learning Repository pelo Dr. Daqing Chen (London South Bank University) e estão hospedados publicamente no Kaggle.

Link público: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data 

### Características do dataset

| Coluna | Tipo esperado | Descrição | Relevância para a tarefa |
|---|---|---|---|
| **Invoice** | string | Número da nota fiscal. Se começar com 'C', é um cancelamento. | **Crucial.** É o identificador da transação (carrinho). Itens com o mesmo *Invoice* foram comprados juntos. |
| **StockCode** | string | Código único do produto. | **Alta.** Usado para identificar de forma única os itens nas regras de associação. |
| **Description** | string | Nome/descrição do produto. | **Alta.** Fundamental para a diretriz de Usabilidade, pois permite traduzir os códigos em nomes legíveis para humanos. |
| **Quantity** | int | Quantidade do produto comprada na transação. | Média. Útil na etapa de limpeza para remover transações inválidas ou negativas. |
| **InvoiceDate** | datetime | Data e hora da transação. | Baixa/Média. Pode ser usada secundariamente para filtrar padrões sazonais. |
| **Price** | float | Preço unitário do produto. | Baixa. Pode ser usado para avaliar o impacto financeiro das regras encontradas. |
| **Customer ID** | float | Identificador único do cliente. | Baixa. Não usaremos o cliente, mas sim a transação. |
| **Country** | string | País de residência do cliente. | Baixa. Pode ser usado futuramente para gerar regras de associação segmentadas por país. |

### Relação com o problema de negócio

Este dataset é o cenário perfeito para algoritmos de Mineração de Padrões Frequentes, pois sua estrutura mapeia exatamente a relação de "múltiplos itens (Description) pertencentes a uma mesma cesta de compras (Invoice)". O alto volume de dados permitirá testar a escalabilidade dos modelos.


## 1.3 Diretrizes selecionadas e hipótese experimental

### Diretrizes selecionadas

| Diretriz | Por que é pertinente ao problema? | Possível efeito esperado na solução da LLM |
|---|---|---|
| **Eficiência e escalabilidade** | O dataset é muito extenso e a tarefa de mineração de padrões frequentes é computacionalmente exaustiva. Algoritmos clássicos podem esgotar a RAM da máquina com facilidade. | Esperamos que a LLM proponha algoritmos otimizados (como o FP-Growth em detrimento do Apriori padrão) e sugira técnicas eficientes de poda de itens raros e otimização de tipos de dados. |
| **Usabilidade e acessibilidade** | As regras geradas pela mineração são baseadas em métricas puramente estatísticas. Para ter valor de negócio, pessoas não-técnicas precisam conseguir ler e entender os resultados. | Esperamos que a LLM sugira gráficos intuitivos , código modular bem documentado e traduza as métricas em *insights* em linguagem de negócios. |
### Hipótese experimental

Esperamos que, ao explicitar as diretrizes de **Eficiência e Escalabilidade** e **Usabilidade e Acessibilidade**, a LLM proponha uma solução mais robusta. Na interação *baseline*, prevemos a sugestão de um pipeline genérico, com algoritmos padrão e resultados em tabelas brutas. Na interação *guiada*, hipotetizamos que a IA recomendará algoritmos de menor custo computacional (ex: FP-Growth), etapas rigorosas de pré-processamento para poupar memória e, sobretudo, a criação de visualizações gráficas didáticas acompanhadas de descrições textuais que traduzam as regras estatísticas em *insights* acionáveis para o público de negócios.

### Evidências que serão analisadas

Para avaliar a hipótese, compararemos as trilhas quanto a:

- algoritmos sugeridos;
- etapas de preparação dos dados;
- parâmetros recomendados;
- métricas propostas;
- justificativas técnicas;
- riscos apontados;
- limitações reconhecidas;
- aderência às diretrizes escolhidas.


## 1.4 Interações com LLM — Business Understanding

### Trilha A — baseline

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]  
**Subtarefa:** obter apoio para formular o Business Understanding.

#### Prompt principal — baseline

```text
[INSERIR PROMPT USADO NA TRILHA A]

Exemplo:
Estou fazendo um trabalho de Mineração de Dados com o dataset [NOME].
A tarefa é [TAREFA].
Ajude-me a formular o Business Understanding, incluindo objetivo do dataset,
origem dos dados, características gerais e relação com o problema.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE DA RESPOSTA. NÃO COPIAR RESPOSTA LONGA INTEGRALMENTE SE NÃO FOR NECESSÁRIO.]

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]  
**Subtarefa:** obter apoio para formular o Business Understanding considerando as diretrizes escolhidas.

#### Prompt principal — guiado

```text
[INSERIR PROMPT USADO NA TRILHA B]

Exemplo:
Estou fazendo um trabalho de Mineração de Dados com o dataset [NOME].
A tarefa é [TAREFA].
Ajude-me a formular o Business Understanding.

Considere explicitamente as seguintes diretrizes de IA responsável:
1. [DIRETRIZ 1]: [EXPLICAR COMO ELA SE APLICA AO PROBLEMA]
2. [DIRETRIZ 2]: [EXPLICAR COMO ELA SE APLICA AO PROBLEMA]

Além da descrição do problema, indique riscos, limitações e critérios que deveriam
orientar a solução de mineração de dados.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE DA RESPOSTA]


## 1.5 Análise crítica das trilhas — Business Understanding

| Critério | Trilha A — baseline | Trilha B — guiada | Diferença observada |
|---|---|---|---|
| Clareza do problema | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Relação com o dataset | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Consideração das diretrizes | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Identificação de riscos | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da justificativa | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Erros ou omissões | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Síntese crítica

[INSERIR ANÁLISE COMPARATIVA]

A resposta guiada deve ser considerada melhor apenas se houver evidência concreta de melhoria. Exemplos de evidência:

- mencionou atributos sensíveis que a baseline ignorou;
- propôs critérios de avaliação mais adequados;
- apontou limitações metodológicas relevantes;
- sugeriu documentação mais rastreável;
- evitou recomendações inadequadas feitas na baseline.

Caso não haja diferença relevante, isso também deve ser relatado.


# 2. Data Understanding & Data Preparation

Nesta seção, o grupo deve explorar o dataset, identificar problemas de qualidade dos dados e preparar os dados para a etapa de modelagem.

Além disso, deve registrar como a LLM foi usada para apoiar:

- exploração inicial;
- identificação de problemas;
- tratamento de valores ausentes;
- transformação de atributos;
- seleção de variáveis;
- preparação específica para a tarefa de mineração;
- identificação de aspectos relacionados às diretrizes de IA responsável.


In [2]:
# Carregamento do dataset
# Esta célula deve ser adaptada de acordo com o formato do arquivo.

def carregar_dataset(caminho: str) -> pd.DataFrame:
    # Carrega um dataset a partir de um arquivo CSV, Excel, JSON ou Parquet.
    # Adapte esta função se o dataset estiver em outro formato.
    if caminho == "[INSERIR_CAMINHO_DO_ARQUIVO]":
        raise ValueError("Substitua DATA_PATH pelo caminho real do dataset.")

    caminho_path = Path(caminho)

    if not caminho_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    if caminho_path.suffix.lower() == ".csv":
        return pd.read_csv(caminho_path)
    elif caminho_path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(caminho_path)
    elif caminho_path.suffix.lower() == ".json":
        return pd.read_json(caminho_path)
    elif caminho_path.suffix.lower() == ".parquet":
        return pd.read_parquet(caminho_path)
    else:
        raise ValueError("Formato não tratado neste modelo. Adapte a função carregar_dataset.")

# Exemplo de uso:
df = carregar_dataset(DATA_PATH)
print("Dataset carregado com sucesso.")
print(f"Número de linhas: {df.shape[0]}")
print(f"Número de colunas: {df.shape[1]}")
display(df.head())

Dataset carregado com sucesso.
Número de linhas: 1067371
Número de colunas: 8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Verificação mínima das restrições do dataset
# A especificação exige dataset público, com pelo menos 1000 linhas e 4 colunas.
# O link público deve ser documentado em Markdown; aqui validamos apenas dimensões.

def verificar_restricoes_dataset(df: pd.DataFrame) -> None:
    n_linhas, n_colunas = df.shape

    print("Verificação das dimensões do dataset")
    print(f"- Linhas: {n_linhas}")
    print(f"- Colunas: {n_colunas}")

    if n_linhas < 1000:
        print("ATENÇÃO: o dataset possui menos de 1000 linhas.")
    else:
        print("OK: o dataset possui pelo menos 1000 linhas.")

    if n_colunas < 4:
        print("ATENÇÃO: o dataset possui menos de 4 colunas.")
    else:
        print("OK: o dataset possui pelo menos 4 colunas.")

# Executar após carregar o dataset:
verificar_restricoes_dataset(df)


Verificação das dimensões do dataset
- Linhas: 1067371
- Colunas: 8
OK: o dataset possui pelo menos 1000 linhas.
OK: o dataset possui pelo menos 4 colunas.


In [4]:
# Exploração inicial
# Esta célula gera uma visão geral do dataset.
# O grupo deve interpretar os resultados em uma célula Markdown logo abaixo.

def resumo_inicial(df: pd.DataFrame) -> pd.DataFrame:
    resumo = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_nulos": df.isna().sum(),
        "perc_nulos": (df.isna().mean() * 100).round(2),
        "n_unicos": df.nunique(dropna=True)
    })

    return resumo.sort_values(by="perc_nulos", ascending=False)

# Exemplo de uso:
resumo = resumo_inicial(df)
display(resumo)


,tipo,n_nulos,perc_nulos,n_unicos
Customer ID,float64,243007,22.77,5942
Description,str,4382,0.41,5698
StockCode,str,0,0.00,5305
Invoice,str,0,0.00,53628
Quantity,int64,0,0.00,1057
InvoiceDate,str,0,0.00,47635
Price,float64,0,0.00,2807
Country,str,0,0.00,43


In [5]:
# Estatísticas descritivas
# O objetivo é separar análise de variáveis numéricas e categóricas.

def estatisticas_descritivas(df: pd.DataFrame):
    numericas = df.select_dtypes(include=np.number)
    categoricas = df.select_dtypes(exclude=np.number)

    print("Colunas numéricas:", list(numericas.columns))
    print("Colunas não numéricas:", list(categoricas.columns))

    if len(numericas.columns) > 0:
        print("\nEstatísticas descritivas — variáveis numéricas")
        display(numericas.describe().T)

    if len(categoricas.columns) > 0:
        print("\nEstatísticas descritivas — variáveis categóricas/textuais")
        display(categoricas.describe().T)

# Exemplo de uso:
estatisticas_descritivas(df)


Colunas numéricas: ['Quantity', 'Price', 'Customer ID']
Colunas não numéricas: ['Invoice', 'StockCode', 'Description', 'InvoiceDate', 'Country']

Estatísticas descritivas — variáveis numéricas


,count,mean,std,min,25%,50%,75%,max
Quantity,1067371.0,9.938898,172.705794,-80995.00,1.00,3.0,10.00,80995.0
Price,1067371.0,4.649388,123.553059,-53594.36,1.25,2.1,4.15,38970.0
Customer ID,824364.0,15324.638504,1697.464450,12346.00,13975.00,15255.0,16797.00,18287.0



Estatísticas descritivas — variáveis categóricas/textuais


,count,unique,top,freq
Invoice,1067371,53628,537434,1350
StockCode,1067371,5305,85123A,5829
Description,1062989,5698,WHITE HANGING HEART T-LIGHT HOLDER,5918
InvoiceDate,1067371,47635,2010-12-06 16:57:00,1350
Country,1067371,43,United Kingdom,981330


## 2.1 Exploração inicial — interpretação

[INSERIR ANÁLISE DA EXPLORAÇÃO INICIAL]

A análise deve comentar, no mínimo:

- quantidade de linhas e colunas;
- tipos de atributos;
- valores ausentes;
- variáveis com muitos valores distintos;
- possíveis identificadores;
- possíveis atributos sensíveis;
- variáveis relevantes para a tarefa de mineração;
- limitações iniciais percebidas.

### Relação com as diretrizes escolhidas

[EXPLICAR COMO A EXPLORAÇÃO INICIAL SE RELACIONA ÀS DIRETRIZES]

Exemplos:

- Se a diretriz for **Privacidade e Segurança**, verificar atributos identificadores ou sensíveis.
- Se a diretriz for **Justiça e Viés**, verificar grupos sub-representados ou classes desbalanceadas.
- Se a diretriz for **Eficiência e Escalabilidade**, verificar dimensionalidade, cardinalidade e volume dos dados.
- Se a diretriz for **Explicabilidade**, verificar se os atributos são interpretáveis.


In [6]:
# Visualizações exploratórias básicas
# O grupo deve adaptar as colunas às características do dataset.

def plot_distribuicao_numerica(df: pd.DataFrame, coluna: str) -> None:
    plt.figure(figsize=(8, 4))
    df[coluna].dropna().hist(bins=30)
    plt.title(f"Distribuição de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()

def plot_contagem_categorica(df: pd.DataFrame, coluna: str, top_n: int = 20) -> None:
    plt.figure(figsize=(10, 4))
    df[coluna].value_counts(dropna=False).head(top_n).plot(kind="bar")
    plt.title(f"Contagem de valores — {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

def plot_matriz_correlacao(df: pd.DataFrame) -> None:
    numericas = df.select_dtypes(include=np.number)

    if numericas.shape[1] < 2:
        print("Não há pelo menos duas variáveis numéricas para calcular correlação.")
        return

    corr = numericas.corr()

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Matriz de correlação — variáveis numéricas")
    plt.tight_layout()
    plt.show()

# Exemplos de uso:
# plot_distribuicao_numerica(df, "[COLUNA_NUMERICA]")
# plot_contagem_categorica(df, "[COLUNA_CATEGORICA]")
# plot_matriz_correlacao(df)


## 2.2 Análise visual — interpretação

[INSERIR INTERPRETAÇÃO DOS GRÁFICOS]

A interpretação deve evitar apenas descrever o óbvio. Procure responder:

- Há concentração de valores?
- Há valores extremos?
- Há categorias muito raras?
- Há desbalanceamento?
- Os padrões observados afetam a tarefa de mineração?
- Alguma visualização sugere risco associado às diretrizes escolhidas?

> Não invente conclusões. Toda afirmação empírica deve estar apoiada em uma tabela, gráfico ou cálculo executado no notebook.


In [7]:
# Detecção simples de outliers em variáveis numéricas usando IQR
# Esta função é um exemplo. O grupo deve avaliar se o critério faz sentido para cada atributo.

def detectar_outliers_iqr(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
    serie = df[coluna].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    print(f"Coluna: {coluna}")
    print(f"Q1: {q1}")
    print(f"Q3: {q3}")
    print(f"IQR: {iqr}")
    print(f"Limite inferior: {limite_inferior}")
    print(f"Limite superior: {limite_superior}")
    print(f"Número de outliers: {len(outliers)}")

    return outliers

# Exemplo de uso:
# outliers = detectar_outliers_iqr(df, "[COLUNA_NUMERICA]")
# display(outliers.head())


## 2.3 Problemas de qualidade dos dados

| Problema identificado | Evidência | Decisão tomada | Justificativa |
|---|---|---|---|
| Valores ausentes em [COLUNA] | [TABELA/GRÁFICO/CÁLCULO] | [REMOVER/IMPUTAR/MANTER] | [JUSTIFICAR] |
| Duplicatas | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Outliers em [COLUNA] | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Categorias raras | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Atributos sensíveis ou identificadores | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |

### Observação sobre responsabilidade

[EXPLICAR SE ALGUMA DECISÃO DE PREPARAÇÃO TEM IMPLICAÇÕES PARA AS DIRETRIZES ESCOLHIDAS]

Exemplo:

> A remoção de um atributo sensível pode reduzir risco de exposição indevida, mas também pode dificultar a análise de disparidades entre grupos. Essa decisão deve ser justificada com cuidado.


In [8]:
def preparar_dados_basico(
    df: pd.DataFrame,
    colunas_remover=None,
    imputar_numericas=False,
    imputar_categoricas=True,
    remover_outliers=True,
    filtrar_itens_raros=True,
    min_suporte_item=0.001
) -> pd.DataFrame:
    df_prep = df.copy()

    # 1. Remover duplicatas exatas
    df_prep = df_prep.drop_duplicates()

    # 2. Remover colunas irrelevantes / identificadores
    if colunas_remover is None:
        colunas_remover = [
            "Customer ID"  # conforme decisão
        ]

    colunas_existentes = [col for col in colunas_remover if col in df_prep.columns]
    df_prep = df_prep.drop(columns=colunas_existentes)

    # 3. Tratamento de valores ausentes
    # Numéricas: não imputar por padrão (evita distorcer lógica transacional)
    if imputar_numericas:
        for col in df_prep.select_dtypes(include=np.number).columns:
            df_prep[col] = df_prep[col].fillna(df_prep[col].median())

    # Categóricas: manter consistência mínima
    if imputar_categoricas:
        for col in df_prep.select_dtypes(include=["object", "category"]).columns:
            df_prep[col] = df_prep[col].fillna("desconhecido")

    # 4. Remoção de outliers relevantes para a tarefa
    if remover_outliers:
        if "Quantity" in df_prep.columns:
            df_prep = df_prep[df_prep["Quantity"] > 0]

        if "Price" in df_prep.columns:
            df_prep = df_prep[df_prep["Price"] > 0]

    # 5. Padronização de descrição
    if "Description" in df_prep.columns:
        df_prep["Description"] = df_prep["Description"].str.strip().str.lower()

    # 6. Filtragem de itens raros (baseado em StockCode)
    if filtrar_itens_raros and "StockCode" in df_prep.columns:
        freq = df_prep["StockCode"].value_counts(normalize=True)
        itens_validos = freq[freq >= min_suporte_item].index
        df_prep = df_prep[df_prep["StockCode"].isin(itens_validos)]

    # 7. Estruturação transacional (Invoice -> lista de itens)
    if "Invoice" in df_prep.columns and "StockCode" in df_prep.columns:
        df_transacoes = (
            df_prep
            .groupby("Invoice")["StockCode"]
            .apply(list)
            .reset_index(name="itens")
        )
        return df_transacoes

    return df_prep


df_prep = preparar_dados_basico(df)
display(df_prep.head())


,Invoice,itens
0,489434,"[21232, 21523]"
1,489436,"[21755, 21754, 84879, 21181, 22111]"
2,489437,"[84970S, 20971, 22111, 22112]"
3,489438,[20711]
4,489439,"[22138, 22139, 22352, 21731, POST]"


## 2.4 Justificativa da preparação dos dados

[INSERIR JUSTIFICATIVA DAS TRANSFORMAÇÕES REALIZADAS]

Explique:

- quais colunas foram removidas e por quê;
- como valores ausentes foram tratados;
- como variáveis categóricas foram tratadas;
- se houve normalização, discretização ou codificação;
- se houve remoção de outliers;
- quais riscos essas decisões introduzem;
- como essas decisões se relacionam às diretrizes escolhidas.

> A preparação dos dados não deve ser apenas uma sequência de comandos. Toda decisão relevante precisa de justificativa técnica.


## 2.5 Interações com LLM — Data Understanding & Data Preparation

### Subtarefa comparável

Nesta etapa, as duas trilhas devem receber a mesma subtarefa geral.

**Subtarefa escolhida:** [EXEMPLO: propor estratégias de limpeza, transformação e preparação dos dados para a tarefa de mineração]

---

### Trilha A — baseline

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]

#### Prompt principal — baseline

```text
[INSERIR PROMPT BASELINE]

Exemplo:
Tenho um dataset com as seguintes colunas: [LISTAR COLUNAS].
A tarefa de mineração é [TAREFA].
Sugira uma estratégia de Data Understanding e Data Preparation para preparar
os dados para essa tarefa.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE]

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** [INSERIR MODELO]  
**Data de acesso:** [INSERIR DATA]  
**Link da conversa:** [INSERIR LINK DA CONVERSA]

#### Prompt principal — guiado

```text
[INSERIR PROMPT GUIADO]

Exemplo:
Tenho um dataset com as seguintes colunas: [LISTAR COLUNAS].
A tarefa de mineração é [TAREFA].
Sugira uma estratégia de Data Understanding e Data Preparation.

Considere explicitamente as diretrizes:
1. [DIRETRIZ 1]
2. [DIRETRIZ 2]

Ao sugerir a preparação, indique:
- possíveis atributos sensíveis, se houver;
- riscos de viés ou perda de informação;
- decisões que devem ser documentadas;
- transformações necessárias;
- limitações e trade-offs.
```

#### Síntese da resposta da LLM

[INSERIR SÍNTESE]


## 2.6 Comparação crítica — Data Understanding & Preparation

| Aspecto | Trilha A — baseline | Trilha B — guiada | Decisão do grupo |
|---|---|---|---|
| Tratamento de nulos | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Tratamento de outliers | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Atributos sensíveis | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Codificação de variáveis | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Redução de dimensionalidade | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Justificativa técnica | [ANALISAR] | [ANALISAR] | [DECISÃO] |
| Riscos e limitações | [ANALISAR] | [ANALISAR] | [DECISÃO] |

### Análise crítica

[INSERIR ANÁLISE COMPARATIVA]

A análise deve deixar claro:

- o que a LLM sugeriu corretamente;
- o que estava incompleto;
- o que estava tecnicamente errado;
- o que foi aceito;
- o que foi rejeitado;
- o que foi corrigido pelo grupo.


# 3. Modeling

Nesta seção, o grupo deve documentar a modelagem proposta com apoio da LLM.

Para o TP de padrões frequentes, a modelagem pode envolver, por exemplo:

- Apriori;
- FP-Growth;
- Eclat;
- regras de associação.

Para outros TPs, substituir pelos algoritmos pertinentes, como algoritmos de agrupamento ou classificação.

O foco da Fase 2 é comparar criticamente as soluções sugeridas pela LLM nas duas trilhas e, quando possível, executar testes preliminares para verificar a viabilidade das sugestões.


## 3.1 Interações com LLM — Modeling

### Subtarefa comparável

**Subtarefa escolhida:** Indicação dos algoritmos mais adequados para a tarefa, parâmetros iniciais recomendados, métricas para avaliar a relevância dos padrões e interpretação dos resultados

---

### Trilha A — baseline

**Modelo utilizado:** ChatGPT-5.3  
**Data de acesso:** 02/05/2026  
**Link da conversa:** https://chatgpt.com/share/69f5defb-caa8-83e9-a327-102eef9ac60b

#### Prompt principal — baseline

```text
Eu estou fazendo um trabalho de Mineração de Dados com o dataset "Online Retail II UCI"(https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data). 
Proponha uma estratégia para a tarefa de Mineração de Padrões Frequentes e Regras de Associação (Market Basket Analysis). 
Indique os algoritmos mais adequados, os parâmetros iniciais recomendados, as métricas de interesse para avaliar a relevância dos padrões encontrados e como interpretar e comparar os resultados.

Exemplo:
Com base no dataset [NOME] e na tarefa [TAREFA], proponha uma estratégia de modelagem.
Indique algoritmos, parâmetros iniciais, métricas de avaliação e como comparar resultados.
```

#### Síntese da resposta da LLM

O LLM estrutura a modelagem de uma maneira simples, fornecendo um passo a passo padrão para a tarefa de mineração de padrões frequentes. A estratégia sugerida envolve o uso de Apriori como algoritmo principal e FP-Growth para comparação, isso sugere que o LLM não levou em consideração as características do dataset, já que o Apriori não é um método eficiente para se lidar com uma base de dados tão grande quanto a escolhida devido à possibilidade de esgotamento de memória.
Além disso, o LLM menciona casos de aplicação válidos, como recomendação conjunta, campanhas e bundles, e sugere visualizações como gráficos de barras e heatmaps. No entanto, a interpretação dos resultados ainda é muito voltada para o cientista de dados, exigindo um entendimento de métricas como o Lift e confidence.


---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** ChatGPT-5.3  
**Data de acesso:** 02/05/2026  
**Link da conversa:** https://chatgpt.com/share/69f5deeb-7ce8-83e9-903a-46b1871cbe1d

#### Prompt principal — guiado

```text
Eu estou fazendo um trabalho de Mineração de Dados com o dataset "Online Retail II UCI" (https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data). 
A tarefa será Mineração de Padrões Frequentes e Extração de Regras de Associação (Market Basket Analysis). Ajude-me a criar uma estratégia técnica e metodológica para a Modelagem do problema, incluindo: 
- Algoritmos adequados: Seleção e justificativa de algoritmos (ex: Apriori, FP-Growth, Eclat, etc.) considerando as características do dataset. 
- Parâmetros iniciais: Sugestão justificada para os limiares de suporte mínimo e confiança mínima. 
- Métricas de avaliação e interesse: Como utilizar métricas avançadas (como Lift, Leverage, Conviction) para filtrar padrões triviais ou redundantes. 
- Riscos metodológicos 
- Trade-offs: Análise da relação entre desempenho/custo computacional, uso de memória e interpretabilidade dos padrões gerados.

Crucial: Você deve seguir estritamente as seguintes diretrizes de IA Responsável: 
1. Eficiência e Escalabilidade: O dataset é gigante. Proponha abordagens, pré-processamentos ou algoritmos que evitem o esgotamento de memória (RAM) e otimizem drasticamente o custo computacional. 
2. Usabilidade e Acessibilidade: O resultado final será lido por diretores de Marketing. Proponha como devemos exibir esses resultados (visualizações, gráficos) e como organizar o notebook para que as regras matemáticas sejam traduzidas de forma didática e acionável. 

Descreva como as diretrizes 1 e 2 influenciaram diretamente suas escolhas técnicas.

Exemplo:
Com base no dataset [NOME] e na tarefa [TAREFA], proponha uma estratégia de modelagem.

Considere explicitamente as diretrizes:
1. [DIRETRIZ 1]
2. [DIRETRIZ 2]

Na resposta, indique:
- algoritmos adequados;
- parâmetros iniciais;
- métricas de avaliação;
- riscos metodológicos;
- impacto das diretrizes nas escolhas;
- trade-offs entre desempenho, interpretabilidade, custo computacional e qualidade da solução.
```

#### Síntese da resposta da LLM

O LLM entrega uma estratégia mais refinada para o contexto do trabalho, tratando os desafios de um grande volume de dados e sugerindo formas de traduzir os aspectos muito técnicos. Isso demostra que o LLM conseguiu seguir as diretrizes propostas. 
A IA elege o FP-Growth como algoritmo principal por sua eficiência computacional, limitando o Apriori apenas para fins comparativos e estabelecendo táticas para evitar o esgotamento de memória. Além disso, estrutura a entrega final para o entendimento de pessoas não técnicas, traduzindo o jargão de ciência de dados para linguagem mais simples e propõe um painel visual focado em ações práticas, como sugestões de kits específicos.  


## 3.2 Comparação das soluções sugeridas pela LLM

| Elemento | Trilha A — baseline | Trilha B — guiada | Diferença concreta? | Comentário do grupo |
|---|---|---|---|---|
| Algoritmos sugeridos | Apriori (principal), FP-Growth (secundário) | FP-Growth (principal), Apriori (secundário) | SIM | Necessidade de eficiencia computacional torna o FP-Growth mais apropriado |
| Parâmetros sugeridos | min_sup=0.1, min_conf=0.3 | min_sup = 0.5 ou 0.01, min_conf=0.35 a 0.5 | SIM | Trilha A não justifica o uso desses parâmetros e a trilha B muda os parametros sugeridos no meio da resposta |
| Métricas sugeridas | Support; Confidence; Lift; Leverage; Conviction | Support; Confidence; Lift; Leverage; Conviction.   | NÃO | Métricas comuns para a tarefa de modelagem de padrões frequentes |
| Preparação exigida | Remover linhas falhas | remover linhas falhas e redução de dimensionalidade | SIM | trilha B sugere uma preparação um pouco mais profunda |
| Interpretabilidade | Algoritmos escolhidos interpretáveis e dá exemplo de interpretação de um resultado | Algoritmos escolhidos interpretáveis | SIM | Trilha A deu instruções diretas de interpretação para métricas como Lift e confidence, enquanto a trilha B deu sugestões de como apresentar os resultados para outras pessoas |
| Custo computacional | Alto | Médio | SIM | Escolha de algoritmos mais eficiente da trilha B devido a base de dados grande |
| Riscos ou limitações | Não descreve | Regras espúrias, sazonalidade, itens ignorados, duplicação semântica | SIM | A falta de consideração por riscos na trilha A indica um plano raso |
| Aderência às diretrizes | Pouco | SIM | SIM | Trilha A tem pouca consideração sobre a eficiência e acessibilidade da modelagem, já a trilha B estruturou seu planejamento ao redor dessas diretrizes |

### Síntese

A trilha A cumpre o requisito básico de apresentar uma estratégia de modelagem de padrões frequentes, mas o faz de forma genérica. Já a Prompt B adapta a teoria às características da tarefa. Enquanto a alternativa A sugere parâmetros iniciais sem muita contextualização sobre o seu impacto, a alternativa B ajusta os parâmetros com uma justificativa clara, porém a LLM ainda se confunde e usa paramêtros diferentes nas seções 4 e 11. Além disso, A trilha B considera problemas do mundo real que a trilha A ignora com a listagem de riscos metodológicos

As diferenças mais clara estão nas diretrizes apontadas pela prompt. A trilha A entrega um roteiro de análise de dados padrão. Enquanto isso, a trilha B foca tanto em como o dado será consumido pelo tomador de decisão, propondo dashboards voltados para conversão de vendas e traduzindo os jargões de ciência de dados em linguagem clara, quanto em garantir a eficiência e escalabilidade da modelagem, ao propor algoritmos eficientes e estratégias de lidar com grandes volumes de dados

## 3.3 Template de modelagem para padrões frequentes

Use esta subseção se o TP atual envolver mineração de padrões frequentes. Caso o TP atual seja de agrupamento ou classificação, esta subseção pode ser removida ou mantida como referência não executada.

> **Atenção:** suporte, confiança e lift devem ser interpretados no contexto do dataset. Valores de suporte muito baixos podem gerar muitas regras pouco úteis; valores muito altos podem ocultar padrões relevantes.


In [9]:
# CELULA MODIFICADA PARA O DATASET ONLINE RETAIL II
item_col = "itens"
def gerar_transacoes_de_coluna(df: pd.DataFrame, transaction_col: str, item_col: str) -> list:
    
    transacoes = df[item_col]
    return transacoes.tolist()

transacoes = gerar_transacoes_de_coluna(df_prep, TRANSACTION_COLUMN, item_col)
print(f"Número de transações: {len(transacoes)}")
print(transacoes[0])

Número de transações: 33661
['21232', '21523']


In [10]:
# Exemplo de execução de Apriori e FP-Growth com mlxtend
# Esta célula é semi-funcional e depende da instalação da biblioteca mlxtend.
# O grupo deve ajustar suporte, confiança e métrica conforme o problema.

def executar_padroes_frequentes(transacoes, min_support=0.05, min_confidence=0.6):
    try:
        from mlxtend.preprocessing import TransactionEncoder
        from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
    except ImportError:
        print("Biblioteca mlxtend não instalada.")
        print("Instale com: pip install mlxtend")
        return None

    te = TransactionEncoder()
    matriz = te.fit(transacoes).transform(transacoes)
    df_transacoes = pd.DataFrame(matriz, columns=te.columns_)

    resultados = {}

    inicio = time.time()
    itemsets_apriori = apriori(
        df_transacoes,
        min_support=min_support,
        use_colnames=True
    )
    regras_apriori = association_rules(
        itemsets_apriori,
        metric="confidence",
        min_threshold=min_confidence
    )
    tempo_apriori = time.time() - inicio

    resultados["Apriori"] = {
        "itemsets": itemsets_apriori,
        "regras": regras_apriori,
        "tempo": tempo_apriori
    }

    inicio = time.time()
    itemsets_fpgrowth = fpgrowth(
        df_transacoes,
        min_support=min_support,
        use_colnames=True
    )
    regras_fpgrowth = association_rules(
        itemsets_fpgrowth,
        metric="confidence",
        min_threshold=min_confidence
    )
    tempo_fpgrowth = time.time() - inicio

    resultados["FP-Growth"] = {
        "itemsets": itemsets_fpgrowth,
        "regras": regras_fpgrowth,
        "tempo": tempo_fpgrowth
    }

    return resultados

# Exemplo de uso:
resultados = executar_padroes_frequentes(
    transacoes,
    min_support=0.01,
    min_confidence=0.3
)

In [11]:
# Resumo dos resultados de mineração de padrões frequentes.
# Esta célula não inventa resultados; ela apenas resume o que foi executado.

def resumir_resultados_padroes(resultados):
    if resultados is None:
        print("Nenhum resultado disponível.")
        return pd.DataFrame()

    linhas = []

    for algoritmo, info in resultados.items():
        itemsets = info["itemsets"]
        regras = info["regras"]

        linhas.append({
            "algoritmo": algoritmo,
            "n_itemsets": len(itemsets),
            "n_regras": len(regras),
            "tempo_segundos": round(info["tempo"], 4),
            "suporte_minimo_observado": itemsets["support"].min() if len(itemsets) > 0 else np.nan,
            "suporte_maximo_observado": itemsets["support"].max() if len(itemsets) > 0 else np.nan,
            "confianca_media": regras["confidence"].mean() if len(regras) > 0 else np.nan,
            "lift_medio": regras["lift"].mean() if len(regras) > 0 and "lift" in regras.columns else np.nan
        })

    return pd.DataFrame(linhas)

# Exemplo de uso:
resumo_modelagem = resumir_resultados_padroes(resultados)
display(resumo_modelagem)


,algoritmo,n_itemsets,n_regras,tempo_segundos,suporte_minimo_observado,suporte_maximo_observado,confianca_media,lift_medio
0,Apriori,1084,1309,3.3657,0.010012,0.159383,0.493168,9.54307
1,FP-Growth,1084,1309,0.8100,0.010012,0.159383,0.493168,9.54307


In [12]:
# Exemplo de variação de parâmetros
# Útil para verificar se os resultados são sensíveis às escolhas de suporte e confiança.

def testar_parametros_padroes(transacoes, suportes, confiancas):
    linhas = []

    for sup in suportes:
        for conf in confiancas:
            resultados = executar_padroes_frequentes(
                transacoes,
                min_support=sup,
                min_confidence=conf
            )

            resumo = resumir_resultados_padroes(resultados)

            if not resumo.empty:
                resumo["min_support"] = sup
                resumo["min_confidence"] = conf
                linhas.append(resumo)

    if len(linhas) == 0:
        return pd.DataFrame()

    return pd.concat(linhas, ignore_index=True)

# Exemplo de uso:
grade_resultados = testar_parametros_padroes(
    transacoes,
    suportes=[0.01, 0.03, 0.05],
    confiancas=[0.35, 0.5, 0.7]
)
display(grade_resultados)

,algoritmo,n_itemsets,n_regras,tempo_segundos,suporte_minimo_observado,suporte_maximo_observado,confianca_media,lift_medio,min_support,min_confidence
0,Apriori,1084,1052,2.1098,0.010012,0.159383,0.534689,10.053673,0.01,0.35
1,FP-Growth,1084,1052,0.8213,0.010012,0.159383,0.534689,10.053673,0.01,0.35
2,Apriori,1084,576,1.9176,0.010012,0.159383,0.630169,10.810008,0.01,0.50
3,FP-Growth,1084,576,0.9035,0.010012,0.159383,0.630169,10.810008,0.01,0.50
4,Apriori,1084,132,4.8653,0.010012,0.159383,0.763583,11.977082,0.01,0.70
5,FP-Growth,1084,132,0.8159,0.010012,0.159383,0.763583,11.977082,0.01,0.70
6,Apriori,179,32,0.3653,0.030035,0.159383,0.523152,7.211858,0.03,0.35
7,FP-Growth,179,32,0.7404,0.030035,0.159383,0.523152,7.211858,0.03,0.35
8,Apriori,179,19,0.4117,0.030035,0.159383,0.601291,8.000330,0.03,0.50
9,FP-Growth,179,19,0.6598,0.030035,0.159383,0.601291,8.000330,0.03,0.50


## 3.6 Resultados preliminares da modelagem

Podemos observar que a geração de regras é muito sensível ao suporte mínimo configurado. Com min_support = 0.01, os algoritmos encontraram 1084 itemsets frequentes. Variando a confiança neste nível de suporte, obtivemos 1052 regras com confiança de 0.35, 576 regras com confiança de 0.5 e 132 regras com confiança de 0.7. 

No entanto, ao aumentar o suporte para 0.03, o número de itemsets caiu drasticamente para 179, gerando um máximo de apenas 32 regras (para confiança de 0.35) e caindo para apenas 4 regras quando a confiança mínima foi ajustada. Elevando o suporte para 0.05, os algoritmos identificaram apenas 48 itemsets e nenhuma regra conseguiu ser formada, independentemente do limite de confiança. Isso indica que os produtos são comprados juntos com uma frequência absoluta relativamente baixa

As regras geradas no suporte de 0.01 possuem um lift médio considerável, variando entre 10 e 12. Isso aponta uma correlação forte entre os itens das regras geradas. A confiança média também foi alta, alcançando até 73,5% nos cenários mais restritivos.

O experimento valida a necessidade das diretrizes de escalabilidade. O algoritmo FP-Growth foi muito mais rápido, Enquanto o Apriori levou até 5 segundos em uma das execuções, o FP-Growth entregou os mesmos resultados em apenas 0.8863 segundos

### Se a tarefa for padrões frequentes

- número de padrões encontrados;
- número de regras geradas;
- suporte, confiança e lift;
- tempo de execução;
- redundância e interpretabilidade das regras.

### Se a tarefa for agrupamento

- número de clusters;
- métricas internas, como Silhouette e Davies-Bouldin;
- tamanho dos grupos;
- interpretação dos clusters;
- estabilidade com diferentes parâmetros.

### Se a tarefa for classificação

- métricas como acurácia, precisão, revocação, F1-score e matriz de confusão;
- análise de classes minoritárias;
- overfitting ou underfitting;
- interpretabilidade e possíveis vieses.

### Atenção

Não conclua que um algoritmo é “melhor” apenas por uma métrica isolada.

Uma comparação adequada deve considerar:

- qualidade da solução;
- interpretabilidade;
- custo computacional;
- estabilidade;
- adequação ao problema;
- relação com as diretrizes escolhidas.


## 3.7 Análise da aderência às diretrizes

| Diretriz | Evidência na Trilha A | Evidência na Trilha B | A Trilha B melhorou? | Justificativa |
|---|---|---|---|---|
| Eficiência e Escalabilidade | Sugeriu o algoritmo Apriori como principal e não estabeleceu táticas focadas em otimização de RAM ou de custo computacional | Elegeu o algoritmo FP-Growth como principal especificamente por sua eficiência computacional e estabeleceu táticas para evitar o esgotamento de memória | SIM | Os testes validaram a trilha B, o FP-Growth executou a tarefa consideravelmente mais rápido que o Apriori e com muito menos necessidade de memória |
| Usabilidade e Acessibilidade | Focou somente na interpretação técnica | Estruturou a entrega para o público-alvo, propondo traduzir jargões técnicos e usar painéis visuais focados em ações práticas | SIM | A Trilha B conseguiu sair da ciência de dados pura para um direcionamento aplicável. |

### Discussão

A explicitação das diretrizes de IA Responsável gerou um impacto na estratégia arquitetada pela LLM. Na Trilha A, foi fornecido um pipeline genérico que falhou em avaliar o contexto ao sugerir o Apriori de forma principal ignorando tanto o gargalo de memória quanto a acessibilidade dos resultados ao propor meios de comunicação de resultados.

Por outro lado, a Trilha B demonstra a importância das diretrizes. A sugestão da IA de priorizar o uso do FP-Growth foi validada nos testes. A Trilha B também priorizou a comunicação dos resultado, ao propor dashboards visuais e a conversão de regras estatísticas em estratégias diretas de agrupamento de produtos.


A resposta deve ser específica. Exemplos de boa análise:

- “A Trilha B sugeriu registrar parâmetros e versões dos experimentos, o que melhora rastreabilidade.”
- “A Trilha B sugeriu avaliar desempenho por subgrupos, o que é pertinente à diretriz de justiça e viés.”
- “A Trilha B recomendou um algoritmo mais interpretável, mas com possível perda de desempenho.”
- “Apesar de mencionar privacidade, a Trilha B não propôs nenhuma alteração concreta no pipeline.”

Evite frases vagas como:

- “A resposta guiada foi mais responsável.”
- “A LLM considerou melhor as diretrizes.”
- “O resultado foi mais ético.”


# 4. Evaluation

Nesta seção, o grupo deve avaliar criticamente:

1. os resultados preliminares obtidos;
2. a utilidade das sugestões da LLM;
3. os erros e limitações das respostas;
4. as diferenças entre a Trilha A e a Trilha B;
5. os trade-offs observados;
6. a contribuição real das diretrizes de IA responsável.


## 4.1 Análise dos resultados

[INSERIR ANÁLISE DOS RESULTADOS]

A análise deve responder:

- Os padrões, clusters, classificações ou resultados encontrados fazem sentido?
- Eles são relevantes para o problema de negócio?
- Há resultados triviais, redundantes ou pouco úteis?
- O desempenho obtido é adequado?
- As métricas usadas são compatíveis com a tarefa?
- Há limitações nos dados ou na modelagem que afetam a conclusão?

> Se os experimentos não produziram bons resultados, isso também é um resultado válido, desde que seja analisado tecnicamente.


## 4.2 Avaliação das sugestões da LLM

| Sugestão da LLM | Trilha | Decisão do grupo | Justificativa |
|---|---|---|---|
| [SUGESTÃO 1] | Baseline | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |
| [SUGESTÃO 2] | Guiada | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |
| [SUGESTÃO 3] | Guiada | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |

### Erros, omissões ou alucinações identificadas

| Problema | Trilha | Por que é um problema? | Como o grupo corrigiu? |
|---|---|---|---|
| [ERRO/OMISSÃO] | [A/B] | [EXPLICAR] | [EXPLICAR] |
| [ERRO/OMISSÃO] | [A/B] | [EXPLICAR] | [EXPLICAR] |

### Comentário crítico

[INSERIR COMENTÁRIO SOBRE A QUALIDADE DAS RESPOSTAS DA LLM]


## 4.3 Comparação final entre as trilhas

| Critério | Trilha A — baseline | Trilha B — guiada | Avaliação crítica |
|---|---|---|---|
| Utilidade prática | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Correção técnica | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Clareza das justificativas | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Aderência às diretrizes | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da modelagem | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da avaliação | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Riscos não tratados | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Conclusão comparativa

[INSERIR CONCLUSÃO]

A conclusão deve indicar se a explicitação das diretrizes:

- produziu mudanças concretas;
- produziu apenas mudanças superficiais;
- não produziu diferença relevante;
- introduziu novos trade-offs;
- ajudou a identificar limitações ou riscos;
- levou o grupo a alterar decisões técnicas.


## 4.4 Trade-offs identificados

| Trade-off | Evidência | Decisão do grupo |
|---|---|---|
| Interpretabilidade × desempenho | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Privacidade × utilidade dos dados | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Custo computacional × qualidade da solução | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Simplicidade × completude da análise | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Justiça/viés × disponibilidade de atributos | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |

### Discussão

[INSERIR DISCUSSÃO SOBRE OS TRADE-OFFS]

Nem todo trade-off estará presente em todo trabalho. O grupo deve discutir apenas os que forem pertinentes ao dataset e à tarefa.


## 4.5 Considerações finais da Fase 2

[INSERIR CONSIDERAÇÕES FINAIS]

A conclusão da Fase 2 deve responder:

1. A LLM ajudou em quais partes do trabalho?
2. Em quais partes a LLM errou ou foi superficial?
3. A Trilha B foi de fato diferente da Trilha A?
4. As diretrizes escolhidas tiveram efeito concreto?
5. Quais sugestões serão levadas para a Fase 3?
6. Quais sugestões serão descartadas?
7. O que ainda precisa ser implementado ou corrigido na versão final?

### Próximos passos para a Fase 3

- [ ] Corrigir limitações identificadas nas respostas da LLM.
- [ ] Consolidar a preparação dos dados.
- [ ] Executar modelagem final.
- [ ] Avaliar resultados de forma mais completa.
- [ ] Integrar análise comparativa baseline × guiada × solução final.
- [ ] Revisar documentação e reprodutibilidade.


# 5. Checklist final da Fase 2

Antes de entregar, verifique se o notebook contém:

- [ ] identificação do grupo;
- [ ] link público do dataset;
- [ ] comprovação de que o dataset atende às restrições mínimas;
- [ ] descrição do problema de negócio;
- [ ] descrição das colunas;
- [ ] diretrizes de IA responsável escolhidas;
- [ ] justificativa das diretrizes;
- [ ] hipótese experimental;
- [ ] Trilha A — baseline;
- [ ] Trilha B — guiada;
- [ ] modelo utilizado em cada trilha;
- [ ] data de acesso em cada trilha;
- [ ] links das conversas;
- [ ] prompts principais;
- [ ] síntese crítica das respostas;
- [ ] comparação entre baseline e guiada;
- [ ] sugestões aceitas, rejeitadas e corrigidas;
- [ ] análise de erros e limitações da LLM;
- [ ] conexão com Business Understanding;
- [ ] conexão com Data Understanding & Preparation;
- [ ] conexão com Modeling;
- [ ] conexão com Evaluation;
- [ ] considerações finais e próximos passos.


# 6. Referências

[INSERIR REFERÊNCIAS USADAS PELO GRUPO]

Exemplos de referências possíveis:

- documentação do dataset;
- materiais da disciplina;
- documentação das bibliotecas usadas;
- referências sobre CRISP-DM;
- referências sobre IA responsável;
- referências sobre algoritmos aplicados.

> As referências devem apoiar decisões técnicas e metodológicas. Não use referências apenas de forma decorativa.


# Apêndice A — Modelo de registro de interações com LLM

Esta seção é opcional, mas recomendada para organizar evidências.

| ID | Trilha | Subtarefa | Modelo | Data | Link | Prompt resumido | Decisão do grupo |
|---|---|---|---|---|---|---|---|
| A1 | Baseline | Business Understanding | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B1 | Guiada | Business Understanding | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| A2 | Baseline | Data Preparation | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B2 | Guiada | Data Preparation | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| A3 | Baseline | Modeling | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B3 | Guiada | Modeling | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |


In [13]:
# Apêndice B — Estrutura opcional para registrar prompts e respostas de forma organizada
# Esta estrutura pode ajudar o grupo a manter rastreabilidade.
# Evite inserir respostas muito longas no notebook; prefira sínteses críticas e links das conversas.

registros_llm = [
    {
        "id": "A1",
        "trilha": "baseline",
        "subtarefa": "Business Understanding",
        "modelo": "[INSERIR MODELO]",
        "data": "[INSERIR DATA]",
        "link_conversa": "[INSERIR LINK DA CONVERSA]",
        "prompt_resumido": "[INSERIR RESUMO DO PROMPT]",
        "sintese_resposta": "[INSERIR SÍNTESE DA RESPOSTA]",
        "decisao_grupo": "[ACEITA/REJEITADA/CORRIGIDA]",
        "comentario_critico": "[INSERIR COMENTÁRIO]"
    },
    {
        "id": "B1",
        "trilha": "guiada",
        "subtarefa": "Business Understanding",
        "modelo": "[INSERIR MODELO]",
        "data": "[INSERIR DATA]",
        "link_conversa": "[INSERIR LINK DA CONVERSA]",
        "prompt_resumido": "[INSERIR RESUMO DO PROMPT COM DIRETRIZES]",
        "sintese_resposta": "[INSERIR SÍNTESE DA RESPOSTA]",
        "decisao_grupo": "[ACEITA/REJEITADA/CORRIGIDA]",
        "comentario_critico": "[INSERIR COMENTÁRIO]"
    }
]

df_registros_llm = pd.DataFrame(registros_llm)
display(df_registros_llm)


,id,trilha,subtarefa,modelo,data,link_conversa,prompt_resumido,sintese_resposta,decisao_grupo,comentario_critico
0,A1,baseline,Business Understanding,[INSERIR MODELO],[INSERIR DATA],[INSERIR LINK DA CONVERSA],[INSERIR RESUMO DO PROMPT],[INSERIR SÍNTESE DA RESPOSTA],[ACEITA/REJEITADA/CORRIGIDA],[INSERIR COMENTÁRIO]
1,B1,guiada,Business Understanding,[INSERIR MODELO],[INSERIR DATA],[INSERIR LINK DA CONVERSA],[INSERIR RESUMO DO PROMPT COM DIRETRIZES],[INSERIR SÍNTESE DA RESPOSTA],[ACEITA/REJEITADA/CORRIGIDA],[INSERIR COMENTÁRIO]
